> 在线讲义：[第 14 章 · 无约束优化：步长、动量、Adam / AdamW、学习率表、高斯-牛顿与 LM](https://wikieden.github.io/math-and-eval-stats/lessons/14-unconstrained-optimization.html) · 本书《数学与实验统计》

# 第 14 章 · 无约束优化：步长、动量、Adam / AdamW、学习率表、高斯-牛顿与 LM

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/14-unconstrained-optimization.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 GD | x − ηHx | `gd(0.18, 500)` | 36 步；lr = 0.21 发散 |
| 第二步 动量 | v = βv + g | `gd(0.05, 2000, 0.5)` | 59 步 |
| 第三步 Adam | Algorithm 1 | `adam_step(...)` | 更新 [0.001, 0.001] |
| 第三步 AdamW | 解耦衰减 | `adam_step(..., wd=0.1, decoupled=True/False)` | 0.9989 / 0.999 |
| 第四步 学习率表 | warmup + cosine | `warmup_cosine(550)` | 1.65e-4 |
| 第五步 GN / LM | (JᵀJ + λI)Δ = −Jᵀr | `gauss_newton()`、`levenberg_marquardt()` | 第 1 步 30636 / 12.7 |

例子：二次函数 f(x) = ½(x₁² + 10x₂²)；Adam / AdamW 的第一步；warmup + cosine；梯度裁剪；高斯-牛顿和 LM 拟合 y = a·e^{bx}。
里程碑 ME-M9 的 Rosenbrock 对比和与 torch.optim.AdamW 对齐留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np
H = np.diag([1.0, 10.0])  # f(x) = ½ xᵀ H x，最小点在 0
X0 = np.array([1.0, 1.0])
GX = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
GY = np.array([2.0, 3.3, 5.4, 9.0, 14.8])  # 编的：大约 y = 2·e^{x}

**函数 `gd`**：梯度下降（可选动量 v ← βv + g，x ← x − lr·v）。返回 (最后的 x, 第一次 ‖x‖ < 1e-3 的步数，没到就 −1)。

In [ ]:
def gd(lr: float, steps: int, momentum: float = 0.0) -> tuple[np.ndarray, int]:
    """梯度下降（可选动量 v ← βv + g，x ← x − lr·v）。返回 (最后的 x, 第一次 ‖x‖ < 1e-3 的步数，没到就 −1)。"""
    x, v, hit = X0.copy(), np.zeros(2), -1
    for k in range(1, steps + 1):
        g = H @ x
        v = momentum * v + g
        x = x - lr * v
        if hit < 0 and np.linalg.norm(x) < 1e-3:
            hit = k
        if not np.isfinite(x).all() or np.linalg.norm(x) > 1e6:
            return x, -1
    return x, hit

**函数 `adam_step`**：Adam Algorithm 1 的一步；wd > 0 时，decoupled=True 是 AdamW（直接衰减权重），False 是把 wd·x 加进梯度（L2）。

In [ ]:
def adam_step(x, g, m, v, t, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, wd=0.0, decoupled=True):
    """Adam Algorithm 1 的一步；wd > 0 时，decoupled=True 是 AdamW（直接衰减权重），False 是把 wd·x 加进梯度（L2）。"""
    if wd and not decoupled:
        g = g + wd * x
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g * g
    mh, vh = m / (1 - b1**t), v / (1 - b2**t)
    x = x - lr * mh / (np.sqrt(vh) + eps)
    if wd and decoupled:
        x = x - lr * wd * x
    return x, m, v

**函数 `warmup_cosine`**：线性 warmup 到 peak，再按余弦降到 floor。

In [ ]:
def warmup_cosine(step: int, total: int = 1000, warmup: int = 100, peak: float = 3e-4, floor: float = 3e-5) -> float:
    """线性 warmup 到 peak，再按余弦降到 floor。"""
    if step < warmup:
        return peak * (step + 1) / warmup
    p = (step - warmup) / (total - warmup)
    return floor + 0.5 * (peak - floor) * (1 + math.cos(math.pi * p))

**函数 `clip_by_norm`**：梯度范数超过 max_norm 就按比例缩小，方向不变。

In [ ]:
def clip_by_norm(g: np.ndarray, max_norm: float) -> np.ndarray:
    """梯度范数超过 max_norm 就按比例缩小，方向不变。"""
    n = np.linalg.norm(g)
    return g * (max_norm / n) if n > max_norm else g

**函数 `gauss_newton`**：拟合 y = a·e^{bx}：每步用残差的雅可比 J 解 (JᵀJ) Δ = −Jᵀr。返回每步的 (a, b, 残差平方和)。

In [ ]:
def gauss_newton(a: float = 1.0, b: float = 0.5, iters: int = 6) -> list[tuple[float, float, float]]:
    """拟合 y = a·e^{bx}：每步用残差的雅可比 J 解 (JᵀJ) Δ = −Jᵀr。返回每步的 (a, b, 残差平方和)。"""
    hist = []
    for _ in range(iters):
        e = np.exp(b * GX)
        r = a * e - GY
        hist.append((a, b, float(r @ r)))
        j = np.column_stack([e, a * GX * e])
        da, db = np.linalg.solve(j.T @ j, -j.T @ r)
        a, b = a + da, b + db
    r = a * np.exp(b * GX) - GY
    hist.append((a, b, float(r @ r)))
    return hist

**函数 `levenberg_marquardt`**：LM：解 (JᵀJ + λI) Δ = −Jᵀr；这一步让残差变小就接受并把 λ 减半，否则拒绝并把 λ 乘 10。

In [ ]:
def levenberg_marquardt(a: float = 1.0, b: float = 0.5, iters: int = 6, lam: float = 1.0) -> list[tuple[float, float, float]]:
    """LM：解 (JᵀJ + λI) Δ = −Jᵀr；这一步让残差变小就接受并把 λ 减半，否则拒绝并把 λ 乘 10。"""
    def sse(a, b):
        r = a * np.exp(b * GX) - GY
        return float(r @ r)
    hist = [(a, b, sse(a, b))]
    for _ in range(iters):
        while True:
            e = np.exp(b * GX)
            r = a * e - GY
            j = np.column_stack([e, a * GX * e])
            da, db = np.linalg.solve(j.T @ j + lam * np.eye(2), -j.T @ r)
            if sse(a + da, b + db) < sse(a, b):
                a, b, lam = a + da, b + db, lam / 2
                break
            lam *= 10
        hist.append((a, b, sse(a, b)))
    return hist

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 梯度下降：步长不能超过 2/L（L = 最大曲率 10）")
for lr in (0.05, 0.18, 0.21):
    x, hit = gd(lr, 500)
    print(f"  lr = {lr}：{'发散' if hit < 0 and np.linalg.norm(x) > 1 else f'{hit} 步收敛到 ‖x‖ < 1e-3'}")
print(f"  条件数 κ = 10：沿平的方向（曲率 1）每步只缩 1 − lr·1")

In [ ]:
print("\n第二步 动量：在窄谷里少走之字")
for beta in (0.0, 0.5, 0.8):
    _, hit = gd(0.05, 2000, beta)
    print(f"  lr = 0.05，β = {beta}：{hit} 步")

In [ ]:
print("\n第三步 Adam 的第一步：每个坐标都走约 lr，和梯度大小无关")
g = np.array([0.1, 10.0])
x1, _, _ = adam_step(np.zeros(2), g, np.zeros(2), np.zeros(2), 1)
print(f"  梯度 {g.tolist()} → 更新量 {np.round(-x1, 6).tolist()}（lr = 1e-3）")
w = np.array([1.0, 1.0])
for dec in (True, False):
    xw, _, _ = adam_step(w, np.array([0.1, 10.0]), np.zeros(2), np.zeros(2), 1, wd=0.1, decoupled=dec)
    print(f"  {'AdamW（解耦）' if dec else 'Adam + L2'}，wd = 0.1：新权重 {np.round(xw, 6).tolist()}")

In [ ]:
print("\n第四步 学习率表与梯度裁剪")
print("  warmup + cosine：", [f"{warmup_cosine(s):.2e}" for s in (0, 50, 99, 100, 550, 999)])
print(f"  梯度 [3, 4]（范数 5）裁到 1：{np.round(clip_by_norm(np.array([3.0, 4.0]), 1.0), 4).tolist()}")

In [ ]:
print("\n第五步 高斯-牛顿：用一阶展开的最小二乘当二阶近似")
for k, (a, b, sse) in enumerate(gauss_newton()):
    print(f"  第 {k} 步：a = {a:.4f}，b = {b:.4f}，残差平方和 {sse:.4f}")
print("  LM（加阻尼 λ，变差就拒绝）：")
for k, (a, b, sse) in enumerate(levenberg_marquardt()):
    print(f"  第 {k} 步：a = {a:.4f}，b = {b:.4f}，残差平方和 {sse:.4f}")

## 3. 代码里才看得到的

1. **偏差修正只在开头重要**：t = 1 时 m 只有 0.1g（小 10 倍）、v 只有 0.001g²（$\sqrt v$ 小约 32 倍），不修正的话第一步会走约 3.2 倍的 lr；几千步后修正项接近 1。
2. **LM 的拒绝循环**：`while True` 里不断加大 λ 直到这一步真的变好；所以 LM 的"一步"可能解了好几次线性方程。
3. **`gd` 里的发散判断**：‖x‖ > 10⁶ 或出现 inf / nan 就提前返回，否则 0.21 的学习率会跑出浮点范围。

## 4. 自己改一改

1. 把 `H` 改成 diag(1, 100)（条件数 100），lr 最大能用多少？lr = 0.019 要多少步？
2. 高斯-牛顿从 a = 1.5、b = 1.0 出发（离答案更近），还会跳过头吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 步长小于二除 L，最陡方向说了算；
> 动量像滚球，来回抵消、一路加速；
> Adam 按二阶矩归一，每步约走 lr；AdamW 衰减拿出来单做；
> 先 warmup 后余弦，裁剪按范数；拟合用高斯-牛顿，怕跳过头就加阻尼成 LM。